# Zomato Restaurant Dataset — Data Preprocessing Pipeline

**Project:** What Makes a Restaurant Successful? — Data-Driven Analysis Based on Zomato Ratings  
**Module:** M2 & M3 — Data Acquisition & Preprocessing  
**Team:** T21 — Wenxuan Wu, Jiarui Zhai, Zilin Wang, Jiayang Li, Yuchen Zhang  

This notebook implements a **fully reproducible** preprocessing pipeline.  
Run all cells in sequence to reproduce the cleaned dataset `zomato_clean_all.csv`.

## Step 1 — Environment Setup & Raw Data Loading

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

# ---------------------------------------------------------------
# Resolve raw data path relative to this notebook's directory
# Works regardless of where the user clones the repository
# ---------------------------------------------------------------
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
RAW_PATH     = os.path.join(NOTEBOOK_DIR, 'data', 'zomato_restaurants.csv')
OUTPUT_PATH  = os.path.join(NOTEBOOK_DIR, 'data', 'zomato_clean_all.csv')

df_raw = pd.read_csv(RAW_PATH)

print('=' * 55)
print(f'  Raw data loaded from: {RAW_PATH}')
print(f'  Shape               : {df_raw.shape}')
print(f'  Columns             : {df_raw.shape[1]}')
print('=' * 55)
df_raw.head(3)

  Raw data loaded from: d:\dachuii\数据科学项目\LZU-INFO442-Zomato-Analysis-main\LZU-INFO442-Zomato-Analysis-main\data\zomato_restaurants.csv
  Shape               : (9552, 23)
  Columns             : 23


,Unnamed: 0,Restaurant ID,Restaurant Name,Country Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,Average Cost for two,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,0,6317637,Le Petit Souffle,Phillipines,162.0000,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",1100.0000,Botswana Pula(P),Yes,No,No,No,3.0000,4.8000,Dark Green,Excellent,314.0000
1,1,6304287,Izakaya Kikufuji,Phillipines,162.0000,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,1200.0000,Botswana Pula(P),Yes,No,No,No,3.0000,4.5000,Dark Green,Excellent,591.0000
2,2,6300002,Heat - Edsa Shangri-La,Phillipines,162.0000,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",4000.0000,Botswana Pula(P),Yes,No,No,No,4.0000,4.4000,Green,Very Good,270.0000


In [2]:
# --- Snapshot: raw null counts & dtypes ---
print('--- Column types & null counts (raw) ---')
info = pd.DataFrame({
    'dtype'     : df_raw.dtypes,
    'null_count': df_raw.isnull().sum(),
    'null_%'    : (df_raw.isnull().sum() / len(df_raw) * 100).round(3)
})
print(info)
print(f'\nTotal null cells: {df_raw.isnull().sum().sum()}')

--- Column types & null counts (raw) ---
                        dtype  null_count  null_%
Unnamed: 0             object           0  0.0000
Restaurant ID          object           0  0.0000
Restaurant Name        object           0  0.0000
Country Name           object           0  0.0000
Country Code          float64           0  0.0000
City                   object           0  0.0000
Address                object           0  0.0000
Locality               object           1  0.0100
Locality Verbose       object           1  0.0100
Longitude              object           1  0.0100
Latitude               object           1  0.0100
Cuisines               object          10  0.1050
Average Cost for two  float64           1  0.0100
Currency               object           1  0.0100
Has Table booking      object           1  0.0100
Has Online delivery    object           1  0.0100
Is delivering now      object           1  0.0100
Switch to order menu   object           2  0.0210
Price ran

## Step 2 — Drop Redundant Index Column

In [3]:
df = df_raw.copy()

# The CSV was exported with a default integer index stored as 'Unnamed: 0'.
# It carries no business meaning and only adds redundant data.
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)
    print('Dropped column: Unnamed: 0')
else:
    print('Column Unnamed: 0 not found — skipping.')

print(f'Shape after drop: {df.shape}')

Dropped column: Unnamed: 0
Shape after drop: (9552, 22)


## Step 3 — Deduplication

In [4]:
n_before = len(df)

# Pass 1: Remove fully identical rows
df = df.drop_duplicates()
n_after_full = len(df)

# Pass 2: Remove rows with duplicate Restaurant ID
# Keep first occurrence as it is typically the most complete record.
df = df.drop_duplicates(subset=['Restaurant ID'], keep='first')
n_after_id = len(df)

print(f'Before deduplication       : {n_before:,} rows')
print(f'After full-row dedup        : {n_after_full:,} rows  (removed {n_before - n_after_full})')
print(f'After Restaurant ID dedup   : {n_after_id:,} rows  (removed {n_after_full - n_after_id})')

Before deduplication       : 9,552 rows
After full-row dedup        : 9,552 rows  (removed 0)
After Restaurant ID dedup   : 9,552 rows  (removed 0)


## Step 4 — Missing Value Handling

In [5]:
# ---------- 4a. Anomalous value correction ----------
# 'Has Table booking' contains one spurious value 'Orange'.
# This is clearly a data entry error; corrected to 'No' before imputation.
anomaly_mask = df['Has Table booking'] == 'Orange'
print(f"Anomalous 'Orange' in Has Table booking: {anomaly_mask.sum()} row(s) → corrected to 'No'")
df['Has Table booking'] = df['Has Table booking'].replace({'Orange': 'No'})

# ---------- 4b. Text / categorical fields ----------
# Fill with 'Unknown' so downstream code does not break on NaN strings.
text_cols = ['Restaurant Name', 'Cuisines', 'Locality', 'Locality Verbose', 'Address']
for col in text_cols:
    n_filled = df[col].isnull().sum()
    df[col] = df[col].fillna('Unknown')
    if n_filled:
        print(f'  Filled {n_filled} nulls in "{col}" with "Unknown"')

# ---------- 4c. Binary yes/no fields ----------
binary_text_cols = ['Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu']
for col in binary_text_cols:
    n_filled = df[col].isnull().sum()
    df[col] = df[col].fillna('No')   # conservative default: assume feature is absent
    if n_filled:
        print(f'  Filled {n_filled} nulls in "{col}" with "No"')

# ---------- 4d. Numerical fields ----------
# Median imputation is robust to the heavy-tailed distribution of cost and votes.
num_cols = ['Average Cost for two', 'Votes']
for col in num_cols:
    n_filled = df[col].isnull().sum()
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    if n_filled:
        print(f'  Imputed {n_filled} nulls in "{col}" with median={median_val:.2f}')

# ---------- 4e. Other categorical fields ----------
df['Currency']    = df['Currency'].fillna('Unknown')
df['Rating color']= df['Rating color'].fillna('White')   # White = 'Not rated' in Zomato color scheme
df['Rating text'] = df['Rating text'].fillna('Not rated')
df['Country Code']= df['Country Code'].fillna(0)
df['Price range'] = df['Price range'].fillna(df['Price range'].median())

# ---------- 4f. Drop rows with missing Aggregate rating ----------
# Rating is the target variable; rows without it have no analytical value.
n_before_drop = len(df)
df = df.dropna(subset=['Aggregate rating'])
print(f'\nDropped {n_before_drop - len(df)} rows with missing Aggregate rating.')
print(f'Shape after missing-value treatment: {df.shape}')
print(f'Remaining nulls: {df.isnull().sum().sum()}')

Anomalous 'Orange' in Has Table booking: 1 row(s) → corrected to 'No'
  Filled 10 nulls in "Cuisines" with "Unknown"
  Filled 1 nulls in "Locality" with "Unknown"
  Filled 1 nulls in "Locality Verbose" with "Unknown"
  Filled 1 nulls in "Has Table booking" with "No"
  Filled 1 nulls in "Has Online delivery" with "No"
  Filled 1 nulls in "Is delivering now" with "No"
  Filled 2 nulls in "Switch to order menu" with "No"
  Imputed 1 nulls in "Average Cost for two" with median=400.00
  Imputed 2 nulls in "Votes" with median=31.00

Dropped 2 rows with missing Aggregate rating.
Shape after missing-value treatment: (9550, 22)
Remaining nulls: 0


## Step 5 — Format Correction & Data Type Conversion

In [6]:
# ---------- 5a. Longitude / Latitude ----------
# Some values are stored as strings with extra single-quotes, e.g. "'121.05'".
# Strip the quotes and convert to float for geospatial operations.
for col in ['Longitude', 'Latitude']:
    df[col] = df[col].astype(str).str.replace("'", '', regex=False).str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Longitude dtype: {df["Longitude"].dtype} | nulls: {df["Longitude"].isnull().sum()}')
print(f'Latitude  dtype: {df["Latitude"].dtype}  | nulls: {df["Latitude"].isnull().sum()}')

# ---------- 5b. Yes/No binary encoding ----------
# Convert text flags to integer 0/1 required by most ML algorithms.
binary_cols = ['Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

print('\nBinary columns converted to 0/1:')
print(df[binary_cols].dtypes)

# ---------- 5c. Ensure numeric types for key fields ----------
df['Aggregate rating']    = pd.to_numeric(df['Aggregate rating'], errors='coerce')
df['Average Cost for two']= pd.to_numeric(df['Average Cost for two'], errors='coerce')
df['Votes']               = pd.to_numeric(df['Votes'], errors='coerce')
df['Price range']         = pd.to_numeric(df['Price range'], errors='coerce')
df['Country Code']        = pd.to_numeric(df['Country Code'], errors='coerce')

print('\nData type conversion complete.')

Longitude dtype: float64 | nulls: 0
Latitude  dtype: float64  | nulls: 0

Binary columns converted to 0/1:
Has Table booking       int64
Has Online delivery     int64
Is delivering now       int64
Switch to order menu    int64
dtype: object

Data type conversion complete.


## Step 6 — Outlier Removal (IQR Method)

In [7]:
def iqr_filter(data: pd.DataFrame, col: str) -> pd.DataFrame:
    """Remove rows outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR] for `col`."""
    Q1, Q3 = data[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_removed = ((data[col] < lower) | (data[col] > upper)).sum()
    print(f'  {col}: bounds=[{lower:.2f}, {upper:.2f}], removed {n_removed} outliers')
    return data[(data[col] >= lower) & (data[col] <= upper)]

n_pre_outlier = len(df)
print(f'Rows before outlier removal: {n_pre_outlier:,}')

df_clean = iqr_filter(df, 'Average Cost for two')
df_clean = iqr_filter(df_clean, 'Aggregate rating')

print(f'Rows after  outlier removal: {len(df_clean):,}  (removed {n_pre_outlier - len(df_clean):,} total)')

Rows before outlier removal: 9,550
  Average Cost for two: bounds=[-425.00, 1375.00], removed 853 outliers
  Aggregate rating: bounds=[0.10, 5.70], removed 2130 outliers
Rows after  outlier removal: 6,567  (removed 2,983 total)


## Step 7 — Feature Engineering

In [8]:
# ---------- 7a. Primary_Cuisine ----------
# Cuisines are stored as comma-separated strings, e.g. 'North Indian, Chinese, Fast Food'.
# Extract the first token as the restaurant's dominant cuisine for grouped analysis.
df_clean = df_clean.copy()   # avoid SettingWithCopyWarning
df_clean['Primary_Cuisine'] = df_clean['Cuisines'].apply(
    lambda x: x.split(',')[0].strip() if pd.notna(x) and x != 'Unknown' else 'Unknown'
)

print('Top 10 Primary_Cuisine:')
print(df_clean['Primary_Cuisine'].value_counts().head(10))

# ---------- 7b. Price_Level ----------
# Discretise Average Cost for two into four ordered tiers.
# Thresholds are based on quartile analysis of the IQR-filtered cost distribution
# and aligned with the project's research hypotheses on price-rating relationships.
#
#   Budget   : cost < 600        (≈ Q1 region)
#   Mid-range: 600 ≤ cost ≤ 1200 (≈ Q2–Q3 region)
#   High-end : 1200 < cost ≤ 3000
#   Luxury   : cost > 3000
#
def price_level(price: float) -> str:
    if price < 600:           return 'Budget'
    elif price <= 1200:       return 'Mid-range'
    elif price <= 3000:       return 'High-end'
    else:                     return 'Luxury'

df_clean['Price_Level'] = df_clean['Average Cost for two'].apply(price_level)

print('\nPrice_Level distribution:')
print(df_clean['Price_Level'].value_counts())

Top 10 Primary_Cuisine:
Primary_Cuisine
North Indian    1944
Chinese          540
Cafe             518
Fast Food        478
Bakery           431
American         242
South Indian     201
Pizza            188
Street Food      160
Italian          159
Name: count, dtype: int64

Price_Level distribution:
Price_Level
Budget       4333
Mid-range    2142
High-end       92
Name: count, dtype: int64


## Step 8 — Export Cleaned Dataset

In [9]:
df_clean.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 55)
print(f'  ✅  Cleaned dataset saved to:')
print(f'       {OUTPUT_PATH}')
print(f'  Final shape : {df_clean.shape}')
print(f'  Null cells  : {df_clean.isnull().sum().sum()}')
print('=' * 55)

  ✅  Cleaned dataset saved to:
       d:\dachuii\数据科学项目\LZU-INFO442-Zomato-Analysis-main\LZU-INFO442-Zomato-Analysis-main\data\zomato_clean_all.csv
  Final shape : (6567, 24)
  Null cells  : 0


## Step 9 — Quick Verification

In [10]:
df_check = pd.read_csv(OUTPUT_PATH)

print('--- Final Data Quality Report ---')
print(f'Shape          : {df_check.shape}')
print(f'Null cells     : {df_check.isnull().sum().sum()}')
print(f'Unique restaurants: {df_check["Restaurant ID"].nunique()}')
print()
print('Numeric summaries:')
print(df_check[['Average Cost for two', 'Aggregate rating', 'Votes']].describe())
print()
print('Price_Level distribution:')
print(df_check['Price_Level'].value_counts())
print()
print('Schema (dtype per column):')
print(df_check.dtypes)

--- Final Data Quality Report ---
Shape          : (6567, 24)
Null cells     : 0
Unique restaurants: 6567

Numeric summaries:
       Average Cost for two  Aggregate rating     Votes
count             6567.0000         6567.0000 6567.0000
mean               473.9641            3.3967  158.5844
std                299.6181            0.5417  366.3046
min                  0.0000            1.8000    4.0000
25%                250.0000            3.0000   16.5000
50%                450.0000            3.4000   50.0000
75%                650.0000            3.8000  150.0000
max               1350.0000            4.9000 9667.0000

Price_Level distribution:
Price_Level
Budget       4333
Mid-range    2142
High-end       92
Name: count, dtype: int64

Schema (dtype per column):
Restaurant ID             int64
Restaurant Name          object
Country Name             object
Country Code            float64
City                     object
Address                  object
Locality                 object